# Credit Card Fraud Detection | EDA + ML Models
This notebook focuses on detecting fraudulent credit card transactions using machine learning,
 will perform data analysis, feature engineering, model building, and evaluation


## load + understand data

In [ ]:
# import libraries 

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/chetanmittal033/credit-card-fraud-data/fraudTest.csv")

df.head()


In [ ]:
# Shape
print("Dataset Shape:", df.shape)

#Info
df.info()

In [ ]:
# Check missing values

df.isnull().sum()

## Target Variable Distribution (Fraud vs Legit)

We analyze the distribution of fraudulent and legitimate transactions
to understand class imbalance.

In [ ]:
# Count fraud vs legit 
df['is_fraud'].value_counts()

In [ ]:
# Percentage distribution
df['is_fraud'].value_counts(normalize=True)*100

In [ ]:
# Visualization
plt.figure(figsize=(6,4))
df['is_fraud'].value_counts().plot(kind='bar')
plt.title("Fraud vs Legit Transactions")
plt.xlabel("Class (0 = Legit, 1 = Fraud)")
plt.ylabel("Count")
plt.show()


In [ ]:
df.head(5)

## Exploratory Data Analysis (EDA)

In [ ]:
# Amount distribution by fraud
plt.figure(figsize=(6,4))

# Plot fraud vs legit amount
df[df['is_fraud']==0]['amt'].hist(bins=50)
df[df['is_fraud']==1]['amt'].hist(bins=50)

plt.title("Transaction Amount vs Fraud")
plt.xlabel("Amount")
plt.ylabel("Count")
plt.show()

### Observation

Transaction amounts are highly right-skewed.
Most transactions involve small amounts, while only a few have very large values.
This suggests that scaling will be required before model training.

### Data Overview
The dataset contains 555,719 transactions with 23 features and no missing values.
Several personal identifier columns are present, which do not contribute to fraud detection and may cause overfitting.
These will be removed during preprocessing.


In [ ]:
# Clean Dataset

drop_cols=['sn','first','last','street','trans_num','cc_num']
df_clean=df.drop(columns=drop_cols)
print(df_clean.shape)

## Feature Engineering

In [ ]:
df_clean['trans_date_trans_time'] = pd.to_datetime(
    df_clean['trans_date_trans_time']
)

In [ ]:
df_clean['hour']=df_clean['trans_date_trans_time'].dt.hour
df_clean['day']=df_clean['trans_date_trans_time'].dt.day
df_clean['month']=df_clean['trans_date_trans_time'].dt.month

In [ ]:
df_clean['dob'] = pd.to_datetime(df_clean['dob'])

df_clean['age'] = (
    df_clean['trans_date_trans_time'].dt.year -
    df_clean['dob'].dt.year
)


In [ ]:
df_clean = df_clean.drop(
    columns=['trans_date_trans_time', 'dob']
)

In [ ]:
df_clean.head()

### Feature Engineering Conclusion

Time-based features and customer age were extracted from transaction timestamps and date of birth to capture behavioral patterns related to fraud.


## Categorical Encoding

In [ ]:
df_clean.select_dtypes(include='object').columns

In [ ]:
df_encoded = pd.get_dummies(
    df_clean,
    drop_first=True
)

### Categorical Encoding Conclusion

Categorical variables were transformed using one-hot encoding to make them suitable for machine learning models.


## Train–Test Split (With Stratification)

In [ ]:
X = df_encoded.drop('is_fraud', axis=1)
y = df_encoded['is_fraud']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
# Check balance after split

print("Train balance:")
print(y_train.value_counts(normalize=True)*100)

print("\nTest balance:")
print(y_test.value_counts(normalize=True)*100)

## Baseline Model (Logistic Regression)

In [ ]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced'
)

log_model.fit(X_train, y_train)


In [ ]:
y_pred = log_model.predict(X_test)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

### Baseline Model Performance

The Logistic Regression model achieved high recall (74%) for fraudulent transactions,
indicating strong ability to detect fraud.
However, precision remains low, resulting in many false positives.
Further optimization is required to improve model reliability.


## Train Random Forest (Baseline)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

In [ ]:
y_pred_rf = rf_model.predict(X_test)

print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

### Random Forest Performance

The Random Forest model improved both recall and precision for fraudulent transactions
compared to Logistic Regression.
It reduced false positives while detecting more fraud cases,
making it more suitable for deployment.


**Baseline + Scaled Model(Logistic Regression)**

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Pipeline: Scaling + Logistic Regression
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(class_weight='balanced', max_iter=1000))
])

# Train
pipe_lr.fit(X_train, y_train)

# Predict
y_pred_scaled = pipe_lr.predict(X_test)

# Evaluation
print(confusion_matrix(y_test, y_pred_scaled))
print(classification_report(y_test, y_pred_scaled))

After applying feature scaling, Logistic Regression achieved higher fraud recall (0.86 vs 0.74),
indicating better ability to detect fraudulent transactions.
This came at the cost of slightly lower accuracy due to increased false positives,
which is acceptable in fraud detection systems.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Pipeline: Scaling + Decision Tree
pipe_dt = Pipeline([
    ('scaler', StandardScaler()),
    ('dt', DecisionTreeClassifier(
        max_depth=12,
        class_weight='balanced',
        random_state=42
    ))
])

# Train
pipe_dt.fit(X_train, y_train)

# Predict
y_pred_dt = pipe_dt.predict(X_test)

# Evaluation
print("Decision Tree (Scaled)")
print(confusion_matrix(y_test, y_pred_dt))
print(classification_report(y_test, y_pred_dt))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Pipeline: Scaling + Random Forest
pipe_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        class_weight='balanced',
        n_jobs=-1,
        random_state=42
    ))
])

# Train
pipe_rf.fit(X_train, y_train)

# Predict
y_pred_rf = pipe_rf.predict(X_test)

# Evaluation
print("Random Forest (Scaled)")
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd

results = []

def add_result(name, y_pred):
    results.append({
        "Model": name,
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred)
    })

add_result("Logistic Regression (Scaled)", y_pred_scaled)
add_result("Decision Tree (Scaled)", y_pred_dt)
add_result("Random Forest (Scaled)", y_pred_rf)

results_df = pd.DataFrame(results)
results_df

## Key Summary

The dataset is highly imbalanced, with fraud cases below 1%, making accuracy unreliable. Models were evaluated using Precision, Recall, and F1-score. Logistic Regression benefited from standardization, showing improved recall. Decision Tree and Random Forest showed similar performance with and without scaling, as tree-based models are not sensitive to feature standardization. The scaled Decision Tree achieved the best balance with the highest F1-score (0.29), precision (17%), and recall (92%).

From a business perspective, models prioritize high recall to minimize missed fraud, even at the cost of higher false positives. This is suitable when manual review systems are available, but threshold tuning and class weighting are recommended to reduce operational overhead.

### Note on SMOTE Usage

Although SMOTE is effective for handling class imbalance,
applying it on this large-scale dataset significantly increased
training time and computational cost.

After oversampling, the dataset size doubled,
making ensemble models like Random Forest slow to train
within Kaggle’s resource limits.

Therefore, threshold tuning and class-weighted models
were preferred as more practical solutions in this case.
